# SASV: locked **eval** (report once)

Runs the official SASV **eval** protocol for:

1. **ECAPA-only**
2. **ECAPA + CM** score-sum (`s_asv + (1 - P_spoof)`)

**Rules**

- Finish tuning on **dev** (`02` / `03`) before this notebook.
- Do **not** change fusion / CM / thresholds after you see eval numbers.
- Prefer one full pass; eval has more trials than dev (~63k) → longer GPU time.

Compare published Baseline1-v2 (ECAPA+AASIST) ~**1.71% SASV-EER** on eval.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "score_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

import torch
from experiment_lib import DEFAULT_LA, DEFAULT_SASV, RUNS_DIR
from score_lib import score_ecapa_trials, score_fused_trials

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("LA exists:", DEFAULT_LA.exists())
print("SASV exists:", DEFAULT_SASV.exists())

cuda: True
NVIDIA GeForce RTX 4060 Laptop GPU
LA exists: True
SASV exists: True


## Locked settings

- `SPLIT` is fixed to **`eval`**
- `MAX_TRIALS = 0` → all eval trials
- Set `CM_BACKEND` to the same CM you chose on **dev** (default `lfcc`)
- Set `RUN_ECAPA_ONLY` / `RUN_FUSED` if you only need one system

In [2]:
SPLIT = "eval"
MAX_TRIALS = 0
CM_BACKEND = "lfcc"   # must match your locked dev choice ("lfcc" or "wavlm")
DEVICE = "cuda"
FORCE_CPU = False

RUN_ECAPA_ONLY = True
RUN_FUSED = True

assert SPLIT == "eval", "This notebook is for locked eval only"

## 1. ECAPA-only (eval)

Writes `runs/ecapa_only_eval/metrics_eval.json`

In [3]:
ecapa_summary = None
if RUN_ECAPA_ONLY:
    ecapa_summary = score_ecapa_trials(
        la_root=DEFAULT_LA,
        sasv_root=DEFAULT_SASV,
        split=SPLIT,
        max_trials=MAX_TRIALS,
        device=DEVICE,
        force_cpu=FORCE_CPU,
        output_dir=RUNS_DIR / f"ecapa_only_{SPLIT}",
    )
    display({
        "system": ecapa_summary["system"],
        "sasv_eer_%": ecapa_summary["sasv_eer_percent"],
        "sv_eer_%": ecapa_summary["sv_eer_percent"],
        "spf_eer_%": ecapa_summary["spf_eer_percent"],
        "n": ecapa_summary["num_scored"],
    })
else:
    print("Skipped ECAPA-only")

Trials: {'target': 5370, 'nontarget': 33327, 'spoof': 63882, 'total': 102579}
Device: cuda
[WARN] webrtc_noise_gain not installed – WebRTC disabled.


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Enrol eval:   0%|          | 0/48 [00:00<?, ?it/s]

Score trials:   0%|          | 0/102579 [00:00<?, ?it/s]

{
  "system": "ecapa_only",
  "split": "eval",
  "max_trials": 0,
  "num_scored": 102579,
  "key_counts": {
    "target": 5370,
    "nontarget": 33327,
    "spoof": 63882,
    "total": 102579
  },
  "device": "cuda",
  "sasv_eer": 0.20670391061452661,
  "sv_eer": 0.007635009310986674,
  "spf_eer": 0.2704517704521629,
  "sasv_eer_percent": 20.670391061452662,
  "sv_eer_percent": 0.7635009310986673,
  "spf_eer_percent": 27.04517704521629,
  "note": "ECAPA-alone: SV-EER usually low, SPF-EER high (spoofs look like the target)."
}


{'system': 'ecapa_only',
 'sasv_eer_%': 20.670391061452662,
 'sv_eer_%': 0.7635009310986673,
 'spf_eer_%': 27.04517704521629,
 'n': 102579}

## 2. ECAPA + CM fusion (eval)

Writes `runs/ecapa_plus_<cm>_eval/metrics_eval.json`

In [4]:
fused_summary = None
if RUN_FUSED:
    fused_summary = score_fused_trials(
        la_root=DEFAULT_LA,
        sasv_root=DEFAULT_SASV,
        split=SPLIT,
        max_trials=MAX_TRIALS,
        device=DEVICE,
        force_cpu=FORCE_CPU,
        cm_backend=CM_BACKEND,
        output_dir=RUNS_DIR / f"ecapa_plus_{CM_BACKEND}_{SPLIT}",
    )
    display({
        "system": fused_summary["system"],
        "sasv_eer_%": fused_summary["sasv_eer_percent"],
        "sv_eer_%": fused_summary["sv_eer_percent"],
        "spf_eer_%": fused_summary["spf_eer_percent"],
        "n": fused_summary["num_scored"],
    })
else:
    print("Skipped fused")

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Trials: {'target': 5370, 'nontarget': 33327, 'spoof': 63882, 'total': 102579} | CM=lfcc


Enrol eval:   0%|          | 0/48 [00:00<?, ?it/s]

Score fused:   0%|          | 0/102579 [00:00<?, ?it/s]

{
  "system": "ecapa_plus_lfcc_sum",
  "split": "eval",
  "max_trials": 0,
  "num_scored": 102579,
  "key_counts": {
    "target": 5370,
    "nontarget": 33327,
    "spoof": 63882,
    "total": 102579
  },
  "device": "cuda",
  "cm_backend": "lfcc",
  "fusion": "s_asv + (1 - p_spoof)",
  "sasv_eer": 0.07127940828446822,
  "sv_eer": 0.01564245810055869,
  "spf_eer": 0.09706959707000613,
  "sasv_eer_percent": 7.127940828446821,
  "sv_eer_percent": 1.5642458100558692,
  "spf_eer_percent": 9.706959707000614
}


{'system': 'ecapa_plus_lfcc_sum',
 'sasv_eer_%': 7.127940828446821,
 'sv_eer_%': 1.5642458100558692,
 'spf_eer_%': 9.706959707000614,
 'n': 102579}

## 3. Report table (eval)

Loads metrics from disk so you can re-run this cell after either scoring step.

In [5]:
rows = []
for label, path in [
    ("ecapa_only", RUNS_DIR / "ecapa_only_eval" / "metrics_eval.json"),
    (f"ecapa_plus_{CM_BACKEND}", RUNS_DIR / f"ecapa_plus_{CM_BACKEND}_eval" / f"metrics_eval.json"),
]:
    if not path.exists():
        print("Missing:", path)
        continue
    m = json.loads(path.read_text(encoding="utf-8"))
    rows.append({
        "system": label,
        "split": m.get("split"),
        "n": m.get("num_scored"),
        "sasv_eer_%": round(m["sasv_eer_percent"], 4),
        "sv_eer_%": round(m["sv_eer_percent"], 4),
        "spf_eer_%": round(m["spf_eer_percent"], 4),
    })

rows
# Optional: also print your locked dev numbers for the write-up
dev_paths = [
    RUNS_DIR / "ecapa_only_dev" / "metrics_dev.json",
    RUNS_DIR / f"ecapa_plus_{CM_BACKEND}_dev" / "metrics_dev.json",
]
print("\nDev (already locked):")
for path in dev_paths:
    if path.exists():
        m = json.loads(path.read_text(encoding="utf-8"))
        print(
            path.parent.name,
            f"SASV={m['sasv_eer_percent']:.4f}%",
            f"SV={m['sv_eer_percent']:.4f}%",
            f"SPF={m['spf_eer_percent']:.4f}%",
        )
    else:
        print("Missing", path)


Dev (already locked):
ecapa_only_dev SASV=15.2291% SV=1.2483% SPF=17.9090%
ecapa_plus_lfcc_dev SASV=1.1438% SV=2.0978% SPF=0.0897%


## Done

Copy the eval EERs into your paper/table. Do not re-tune on these numbers.

Reference (published eval): ECAPA alone ~23.8% SASV-EER; B1-v2 (ECAPA+AASIST) ~1.71% SASV-EER.